In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv

# 1. CONFIGURACION INICIAL
USE_BANK_FILES = True  # Cambiar a False para no usar archivos de extractos bancarios
ENV_FILE = os.getenv("RECON_ENV_FILE", "env.txt")
env_path = ENV_FILE if os.path.isabs(ENV_FILE) else os.path.join(os.getcwd(), ENV_FILE)
load_dotenv(env_path, override=False)

BASE_DIR = os.getenv("RECON_ROOT", "")
OUTPUT_BASE_DIR = os.getenv("RECON_OUTPUT_DIR", "Automatic reconciliation")
RECON_IS_S3 = os.getenv("RECON_IS_S3", "false")
IS_S3 = RECON_IS_S3.lower() in {"1", "true", "yes"}
S3FS = None
if IS_S3:
    import s3fs
    S3FS = s3fs.S3FileSystem()

CONFIGS = [
    {
        "sap_file": "SAP Exports/NOV 2025.xlsx",
        "bank_file": "EB_111367_111368.xlsx",
        "account": "111367-111368",
        "month_name": "November",
        "month_number": "11",
        "year": "2025",
    },
    {
        "sap_file": "SAP Exports/DEC 2025.xlsx",
        "bank_file": "EB_111367_111368.xlsx",
        "account": "111367-111368",
        "month_name": "December",
        "month_number": "12",
        "year": "2025",
    },
    {
        "sap_file": "SAP Exports/JAN 2026.xlsx",
        "bank_file": "EB_111367_111368.xlsx",
        "account": "111367-111368",
        "month_name": "January",
        "month_number": "1",
        "year": "2026",
    },
]

COLUMNAS_SALIDA = [
    'Company Code',
    'G/L Account',
    'Journal Entry Type',
    'Posting Key',
    'Journal Entry',
    'Journal Entry Item Text',
    'Posting Date',
    'Value date',
    'Amount in Transaction Currency',
    'Amount in Global Currency',
    'Clearing Date',
    'Clearing Journal Entry',
    'Description'
]

def join_path(base, *parts):
    if IS_S3:
        base_clean = base.rstrip("/")
        tail = "/".join(p.strip("/") for p in parts if p is not None)
        return f"{base_clean}/{tail}" if tail else base_clean
    if base:
        return os.path.join(base, *parts)
    return os.path.join(*parts)

def path_exists(path):
    return S3FS.exists(path) if IS_S3 else os.path.exists(path)

def read_excel_path(path):
    if IS_S3:
        with S3FS.open(path, "rb") as f:
            return pd.read_excel(f)
    return pd.read_excel(path)

def clean_amount(val):
    if pd.isna(val): return 0.0
    if isinstance(val, (int, float)): return float(val)
    val = str(val).replace('JMD', '').replace('USD', '').replace(',', '').strip()
    try: return float(val)
    except: return 0.0

def build_desc_lookup(df_bank):
    desc_col = 'DESCRIPTION'
    debits_col = 'DEBITS'
    credits_col = 'CREDITS'
    if desc_col not in df_bank.columns or (debits_col not in df_bank.columns and credits_col not in df_bank.columns):
        return pd.Series(dtype='object')
    parts = []
    if debits_col in df_bank.columns:
        deb = df_bank[[debits_col, desc_col]].copy()
        deb['amount_match'] = deb[debits_col].apply(clean_amount).abs()
        parts.append(deb[['amount_match', desc_col]])
    if credits_col in df_bank.columns:
        cred = df_bank[[credits_col, desc_col]].copy()
        cred['amount_match'] = cred[credits_col].apply(clean_amount).abs()
        parts.append(cred[['amount_match', desc_col]])
    if not parts:
        return pd.Series(dtype='object')
    eb_long = pd.concat(parts, ignore_index=True)
    eb_long = eb_long[eb_long['amount_match'] != 0]
    eb_long = eb_long.dropna(subset=[desc_col])
    return eb_long.groupby('amount_match')[desc_col].first()

def clean_date(val):
    if pd.isna(val) or val == "" or str(val).lower() == "nan": return ""
    try:
        return pd.to_datetime(val).strftime('%d/%m/%Y')
    except:
        return str(val)

def match_by_amount_list(df1, df2):
    """Cruza df1 y df2 por monto absoluto uno a uno."""
    matched = []
    df1_rem = df1.copy()
    df2_rem = df2.copy()
    
    indices_df1 = df1_rem.index.tolist()
    for idx in indices_df1:
        if idx not in df1_rem.index: continue
        row = df1_rem.loc[idx]
        amt = abs(row['Amount in Transaction Currency'])
        mask = df2_rem['Amount in Transaction Currency'].abs().round(2) == round(amt, 2)
        match_idx = df2_rem[mask].index
        if not match_idx.empty:
            match_row = df2_rem.loc[match_idx[0]]
            matched.append(row.to_dict())
            matched.append(match_row.to_dict())
            df2_rem = df2_rem.drop(match_idx[0])
            df1_rem = df1_rem.drop(idx)
            
    return matched, df1_rem, df2_rem

# 2. PROCESAMIENTO
for idx, config in enumerate(CONFIGS, start=1):
    ARCHIVO_SAP = join_path(BASE_DIR, config["sap_file"])
    CUENTA_OBJETIVO = config["account"]
    MES_NOMBRE = config["month_name"]
    MES_NUMERO = int(config["month_number"])
    ANIO = int(config["year"])
    month_folder = f"{MES_NOMBRE[:3].upper()} {ANIO}"
    ARCHIVO_BANCARIO = join_path(
        BASE_DIR, "Bank Statements", month_folder, config["bank_file"]
    )
    OUTPUT_DIR = join_path(OUTPUT_BASE_DIR, CUENTA_OBJETIVO)
    if not IS_S3:
        os.makedirs(OUTPUT_DIR, exist_ok=True)

    if not path_exists(ARCHIVO_SAP):
        print(f"Saltando: SAP no encontrado: {ARCHIVO_SAP}")
        continue 

    df_original = read_excel_path(ARCHIVO_SAP)
    
    if USE_BANK_FILES and path_exists(ARCHIVO_BANCARIO):
        eb = read_excel_path(ARCHIVO_BANCARIO)
        desc_lookup = build_desc_lookup(eb)
        df_original['Description'] = df_original['Amount in Transaction Currency'].abs().map(desc_lookup).fillna("")
    else:
        df_original['Description'] = ""

    cols_monto = ['Amount in Transaction Currency', 'Amount in Global Currency', 'Amount in Company Code Currency']
    for col in cols_monto:
        if col in df_original.columns:
            df_original[col] = df_original[col].apply(clean_amount)

    # Asegurar tipos para filtros
    df_original['G/L Account'] = df_original['G/L Account'].astype(str)
    df_original['Journal Entry Type'] = df_original['Journal Entry Type'].astype(str)
    df_original['Journal Entry Item Text'] = df_original['Journal Entry Item Text'].astype(str).fillna("")
    
    # Funciones auxiliares con str.contains para flexibilidad
    def filter_acc(df, acc_num):
        return df[df['G/L Account'].str.contains(acc_num, na=False)].copy()
        
    def filter_type(df, type_pattern):
        return df[df['Journal Entry Type'].str.contains(type_pattern, na=False, case=False)]
        
    def filter_desc(df, pattern):
        mask = (df['Description'].str.contains(pattern, case=False, na=False)) | \
               (df['Journal Entry Item Text'].str.contains(pattern, case=False, na=False))
        return df[mask]

    output_file = join_path(OUTPUT_DIR, f"{CUENTA_OBJETIVO}_{MES_NOMBRE}_{ANIO}.xlsx")
    writer = pd.ExcelWriter(output_file, engine='xlsxwriter')
    workbook = writer.book
    fmt_jam = workbook.add_format({'num_format': '#,##0.00 "JMD"'}) 
    fmt_std = workbook.add_format({'num_format': '#,##0.00'})

    # --- PESTAÑA 1: TOP UP ---
    df_topup_acc = filter_acc(df_original, '11136820')
    dz_all = filter_type(df_topup_acc, 'DZ')
    dz_neg = dz_all[dz_all['Amount in Transaction Currency'] < 0].copy()
    others_topup = filter_desc(filter_type(df_topup_acc, 'ZB|ZR|BR'), 'Top Up|Ftop')
    
    matched_topup, _, _ = match_by_amount_list(dz_neg, others_topup)
    df_sheet_topup = pd.DataFrame(matched_topup)
    
    # --- PESTAÑA 2: CASHIER 3 ---
    df_c3_880 = filter_acc(df_original, '11136880')
    dz_all_880 = filter_type(df_c3_880, 'DZ')
    dz_pos_880 = dz_all_880[dz_all_880['Amount in Transaction Currency'] > 0].copy()
    bill_pay_880 = filter_desc(filter_type(df_c3_880, 'ZB|ZR|BR'), 'BILL PAYMENT')
    
    matched_c3a, _, rem_bill_880 = match_by_amount_list(dz_pos_880, bill_pay_880)
    
    df_c3_780 = filter_acc(df_original, '11136780')
    xq_transfer = df_c3_780[df_c3_780['Journal Entry Item Text'].str.contains('BA\\|0760\\||BA\\|0766\\||BA\\|0762\\|', na=False)]
    total_transfer_ab = xq_transfer['Amount in Transaction Currency'].sum()
    
    xq_0003 = df_c3_780[df_c3_780['Journal Entry Item Text'].str.contains('BA\\|0003\\|', na=False)]
    
    ib_ncb = filter_desc(filter_type(df_c3_780, 'IB'), 'NCB MONIES').copy()
    if 'Value date' in ib_ncb.columns:
        ib_ncb['temp_date'] = pd.to_datetime(ib_ncb['Value date'], errors='coerce')
        ib_ncb = ib_ncb[(ib_ncb['temp_date'].dt.month == MES_NUMERO) & (ib_ncb['temp_date'].dt.year == ANIO)]
    
    compensadores_xq = pd.concat([rem_bill_880, ib_ncb], ignore_index=True)
    matched_c3c, _, _ = match_by_amount_list(xq_0003, compensadores_xq)
    
    c3_rows = matched_c3a + matched_c3c + [{}, {"Description": f"TOTAL TRANSFERENCIA AB (BA|0760, 0766, 0762): {total_transfer_ab:,.2f}"}]
    df_sheet_c3 = pd.DataFrame(c3_rows)

    # --- PESTAÑA 3: LIBERATE ---
    xq_liberate = df_c3_780[
        (df_c3_780['Journal Entry Type'].str.contains('XQ', case=False, na=False)) & 
        (~df_c3_780['Journal Entry Item Text'].str.contains('BA\\|0760\\||BA\\|0766\\||BA\\|0762\\||BA\\|0003\\||AJ\\|2247\\|', na=False))
    ]
    
    df_6280 = filter_acc(df_original, '11136280')
    edc_pay = filter_desc(filter_type(df_original, 'ZR|ZB|BR'), 'EDC PAYMENT')
    
    compensadores_lib = pd.concat([df_6280, edc_pay], ignore_index=True)
    matched_lib, _, _ = match_by_amount_list(xq_liberate, compensadores_lib)
    
    df_sheet_lib = pd.DataFrame(matched_lib)

    # Finalizar hojas
    for sheet_name, df_sheet in [("Top Up", df_sheet_topup), ("Cashier 3", df_sheet_c3), ("Liberate", df_sheet_lib)]:
        if df_sheet.empty:
            df_sheet = pd.DataFrame(columns=COLUMNAS_SALIDA)
        
        for date_col in ['Posting Date', 'Value date', 'Clearing Date']:
            if date_col in df_sheet.columns:
                df_sheet[date_col] = df_sheet[date_col].apply(clean_date)
        
        for col in COLUMNAS_SALIDA:
            if col not in df_sheet.columns: df_sheet[col] = ""
        df_sheet = df_sheet[COLUMNAS_SALIDA]
        
        df_sheet.to_excel(writer, sheet_name=sheet_name, index=False)
        worksheet = writer.sheets[sheet_name]
        for i, col in enumerate(df_sheet.columns):
            col_len = df_sheet[col].map(lambda x: len(str(x)) if pd.notna(x) else 0).max()
            max_len = max(col_len, len(str(col))) + 2
            if col in ['Amount in Transaction Currency', 'Amount in Company Code Currency']:
                worksheet.set_column(i, i, max_len, fmt_jam)
            else:
                worksheet.set_column(i, i, max_len)

    writer.close()
    print(f"Reporte Maestro Generado: {output_file}")


Reporte Maestro Generado: Automatic reconciliation/111367-111368/111367-111368_November_2025.xlsx
Reporte Maestro Generado: Automatic reconciliation/111367-111368/111367-111368_December_2025.xlsx
Reporte Maestro Generado: Automatic reconciliation/111367-111368/111367-111368_January_2026.xlsx
